# Transformer Tutorial — 처음부터 만들어 보는 Transformer Encoder

**SMWU Deep Learning 2026 · Lecture 13 (RNN & Transformer) 실습**

이 노트북에서는 강의에서 배운 Transformer의 구성 요소를 PyTorch로 직접 구현합니다.

1. Scaled Dot-Product Attention
2. Self-Attention은 순서를 모른다 (permutation equivariance)
3. Positional Encoding (sinusoidal)
4. Multi-Head Attention
5. Encoder Block (Multi-Head Attention + Add & Norm + FFN)
6. 실습 과제: **숫자 정렬 (sorting)** — Positional Encoding이 있을 때와 없을 때 비교
7. Attention map 시각화
8. PyTorch 내장 `nn.TransformerEncoderLayer`와 비교

> Colab: **런타임 → 런타임 유형 변경 → GPU (T4)** 를 선택하면 빠르지만, CPU에서도 몇 분 안에 끝납니다.

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", device)

## 1. Scaled Dot-Product Attention

강의 슬라이드의 식 그대로입니다.

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

- $Q$ (query): "내가 찾고 싶은 것", $K$ (key): "내가 가진 정보의 이름표", $V$ (value): "실제 정보"
- $\sqrt{d_k}$로 나누는 이유: 차원이 커지면 내적 값이 커져서 softmax가 한쪽으로 쏠리는(saturation) 것을 막기 위해
- `mask`: 보면 안 되는 위치(예: decoder의 미래 단어, padding)를 $-\infty$로 만들어 softmax 후 0이 되게 함

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Q, K, V: (..., seq_len, d_k).  mask: True = 가려야 하는 위치."""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)      # (..., L_q, L_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)                    # 각 행의 합 = 1
    return weights @ V, weights


# 단어 3개 ("나는", "학생", "입니다"), 각 단어는 4차원 벡터
X = torch.randn(3, 4)
W_Q, W_K, W_V = torch.randn(4, 4), torch.randn(4, 4), torch.randn(4, 4)
Q, K, V = X @ W_Q, X @ W_K, X @ W_V

Z, A = scaled_dot_product_attention(Q, K, V)
print("attention weights (각 행의 합 = 1):\n", A.round(decimals=3))
print("row sums:", A.sum(-1))
print("output Z shape:", Z.shape)

## 2. Self-Attention은 순서를 모른다

입력 단어의 순서를 섞으면(permutation $P$), 출력도 **똑같이 섞이기만** 합니다.

$$\mathrm{SelfAttn}(PX) = P\,\mathrm{SelfAttn}(X)$$

즉 "dog bites man"과 "man bites dog"이 모델에게는 같은 단어 집합(set)으로 보입니다.
직접 확인해 봅시다.

In [ ]:
def self_attn(X):
    return scaled_dot_product_attention(X @ W_Q, X @ W_K, X @ W_V)[0]

perm = torch.tensor([2, 0, 1])           # 순서 섞기
out_then_perm = self_attn(X)[perm]       # P · SelfAttn(X)
perm_then_out = self_attn(X[perm])       # SelfAttn(P X)
print("SelfAttn(PX) == P SelfAttn(X) ?", torch.allclose(out_then_perm, perm_then_out, atol=1e-6))

## 3. Positional Encoding (sinusoidal)

순서 정보를 넣어 주기 위해 각 위치 $pos$마다 고정된 벡터를 embedding에 **더합니다**.

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \qquad
  PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

- 낮은 차원: 빠르게 진동 (high frequency), 높은 차원: 천천히 진동 (low frequency)
- 학습 파라미터가 없음 (`register_buffer`로 저장 → `model.to(device)` 시 같이 이동)

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)                        # (max_len, 1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)     # 짝수 차원
        pe[:, 1::2] = torch.cos(pos * div)     # 홀수 차원
        self.register_buffer("pe", pe)

    def forward(self, x):                      # x: (batch, seq_len, d_model)
        return x + self.pe[: x.size(1)]


pe = SinusoidalPositionalEncoding(d_model=128, max_len=60).pe
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
im = axes[0].imshow(pe, cmap="RdBu_r", aspect="auto", vmin=-1, vmax=1)
axes[0].set(xlabel="embedding dimension", ylabel="position", title="Positional Encoding matrix")
fig.colorbar(im, ax=axes[0])
im = axes[1].imshow(pe @ pe.T, cmap="viridis")
axes[1].set(xlabel="position q", ylabel="position p", title=r"$PE_p \cdot PE_q$  (depends only on |p-q|)")
fig.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.show()

**확인:** Positional Encoding을 더하면 permutation equivariance가 깨집니다 → 순서를 구별할 수 있음.

In [ ]:
pe3 = SinusoidalPositionalEncoding(d_model=4, max_len=3).pe
a = self_attn(X + pe3)[perm]
b = self_attn(X[perm] + pe3)
print("with PE, SelfAttn(PX) == P SelfAttn(X) ?", torch.allclose(a, b, atol=1e-6))

## 4. Multi-Head Attention

$d_{model}$ 차원을 $h$개의 head로 나누어 ($d_k = d_{model}/h$) 각 head가 서로 다른
관계(representation subspace)를 보도록 한 뒤, 결과를 concat 하고 $W^O$로 섞습니다.

$$\mathrm{MultiHead}(X) = \mathrm{Concat}(Z_0, \dots, Z_{h-1})\,W^O$$

텐서 모양 변화: `(B, L, d_model)` → `(B, h, L, d_k)` → attention → `(B, L, d_model)`

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split(self, x):                       # (B, L, d_model) -> (B, h, L, d_k)
        B, L, _ = x.shape
        return x.view(B, L, self.h, self.d_k).transpose(1, 2)

    def forward(self, x, mask=None):
        Q, K, V = self.split(self.W_q(x)), self.split(self.W_k(x)), self.split(self.W_v(x))
        Z, attn = scaled_dot_product_attention(Q, K, V, mask)    # Z: (B, h, L, d_k)
        B, _, L, _ = Z.shape
        Z = Z.transpose(1, 2).reshape(B, L, self.h * self.d_k)   # concat heads
        return self.W_o(Z), attn                                 # attn: (B, h, L, L)


mha = MultiHeadAttention(d_model=32, n_heads=4)
out, attn = mha(torch.randn(2, 7, 32))
print("output:", out.shape, "| attention:", attn.shape)

## 5. Encoder Block

강의의 Encoder 그림 그대로:

```
x ─► Multi-Head Self-Attention ─► (+ x) ─► LayerNorm ─► FFN ─► (+) ─► LayerNorm ─► out
     └──────── residual ─────────┘                   └── residual ─┘
```

- **Add & Norm**: `LayerNorm(x + Sublayer(x))` (residual connection 덕분에 깊게 쌓아도 학습이 잘 됨)
- **FFN**: 위치마다 독립적으로 적용되는 2층 MLP, `d_model → d_ff → d_model`

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        a, attn = self.attn(x, mask)
        x = self.norm1(x + self.drop(a))            # Add & Norm
        x = self.norm2(x + self.drop(self.ffn(x)))  # Add & Norm
        return x, attn


class TransformerEncoder(nn.Module):
    """Embedding (+ Positional Encoding) -> N x EncoderBlock -> 위치별 분류기."""

    def __init__(self, vocab_size, d_model=64, n_heads=4, d_ff=128, n_layers=2,
                 n_out=10, use_pe=True, max_len=128):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = SinusoidalPositionalEncoding(d_model, max_len) if use_pe else nn.Identity()
        self.blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.head = nn.Linear(d_model, n_out)

    def forward(self, tokens):
        x = self.embed(tokens) * math.sqrt(self.d_model)   # 논문처럼 embedding에 sqrt(d_model) 곱함
        x = self.pos(x)
        attns = []
        for blk in self.blocks:
            x, a = blk(x)
            attns.append(a)
        return self.head(x), attns


model = TransformerEncoder(vocab_size=10)
logits, attns = model(torch.randint(0, 10, (2, 8)))
print("logits:", logits.shape, "| #params:", sum(p.numel() for p in model.parameters()))

## 6. 실습 과제: 숫자 정렬 (Sorting)

입력: 0–9 숫자 10개 (예: `3 9 1 1 7 0 4 2 8 5`) → 출력: 정렬된 수열 (`0 1 1 2 3 4 5 7 8 9`)

각 출력 위치 $i$는 "$i$번째로 작은 수"를 맞혀야 합니다.
같은 입력 집합이라도 **위치마다 다른 답**을 내야 하므로, 모델은 자기가 몇 번째 위치인지 알아야 합니다.

→ **Positional Encoding이 없으면 이 문제를 풀 수 없습니다.** 두 모델을 같은 조건으로 학습해 비교합니다.

In [ ]:
SEQ_LEN, VOCAB = 10, 10

def make_batch(batch_size):
    x = torch.randint(0, VOCAB, (batch_size, SEQ_LEN))
    y, _ = torch.sort(x, dim=1)
    return x.to(device), y.to(device)

x, y = make_batch(3)
for a, b in zip(x.tolist(), y.tolist()):
    print("input :", a, "\ntarget:", b, "\n")

In [ ]:
def train(use_pe, steps=1500, batch_size=256, lr=1e-3, log_every=100):
    torch.manual_seed(0)
    model = TransformerEncoder(VOCAB, d_model=64, n_heads=4, d_ff=128, n_layers=2,
                               n_out=VOCAB, use_pe=use_pe).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    t0 = time.time()
    for step in range(1, steps + 1):
        model.train()
        x, y = make_batch(batch_size)
        logits, _ = model(x)                                   # (B, L, VOCAB)
        loss = F.cross_entropy(logits.reshape(-1, VOCAB), y.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % log_every == 0:
            acc_tok, acc_seq = evaluate(model)
            hist.append((step, loss.item(), acc_tok, acc_seq))
            print(f"[PE={use_pe!s:5}] step {step:4d}  loss {loss.item():.3f}  "
                  f"token acc {acc_tok:.3f}  sequence acc {acc_seq:.3f}  ({time.time() - t0:.0f}s)")
    return model, np.array(hist)


@torch.no_grad()
def evaluate(model, n=2000):
    model.eval()
    x, y = make_batch(n)
    pred = model(x)[0].argmax(-1)
    return (pred == y).float().mean().item(), (pred == y).all(1).float().mean().item()


model_pe, hist_pe = train(use_pe=True)
model_nope, hist_nope = train(use_pe=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for h, name in [(hist_pe, "with positional encoding"), (hist_nope, "without positional encoding")]:
    axes[0].plot(h[:, 0], h[:, 2], marker="o", label=name)
    axes[1].plot(h[:, 0], h[:, 3], marker="o", label=name)
axes[0].set(xlabel="training step", ylabel="accuracy", title="token accuracy", ylim=(0, 1.02))
axes[1].set(xlabel="training step", ylabel="accuracy", title="whole-sequence accuracy", ylim=(0, 1.02))
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**결과 해석**

- **PE 있음**: 대부분의 위치를 맞히고, 수열 전체를 완벽히 정렬하는 비율도 빠르게 올라갑니다.
- **PE 없음**: token accuracy가 낮은 수준에서 멈춥니다. 위치 정보가 없으므로 같은 숫자 토큰은 어느 위치에 있든
  **항상 같은 출력**을 내게 되어 (permutation equivariance) "i번째로 작은 수"를 표현할 수 없습니다.
  그래서 모델은 위치와 상관없이 "가장 무난한 답"(중앙값 근처, 예: 모두 5)을 내는 쪽으로 수렴합니다.

직접 예측해 보기:

In [ ]:
@torch.no_grad()
def show_prediction(model, seq):
    model.eval()
    x = torch.tensor([seq], device=device)
    pred = model(x)[0].argmax(-1)[0].tolist()
    print("input :", seq)
    print("sorted:", sorted(seq))
    print("pred  :", pred, "✓" if pred == sorted(seq) else "✗")

test_seq = [7, 2, 9, 0, 5, 5, 1, 8, 3, 6]
print("== with PE ==");    show_prediction(model_pe, test_seq)
print("== without PE =="); show_prediction(model_nope, test_seq)

## 7. Attention Map 시각화

학습된 모델이 출력 위치 $i$를 만들 때 어떤 입력을 보고 있는지 확인합니다.
행(row) = query 위치 (출력), 열(column) = key 위치 (입력). 마지막 layer의 head들을 그립니다.

정렬 문제에서는 "출력 $i$번째 = $i$번째로 작은 수"이므로, 잘 학습된 head는 그 값을 가진 입력 위치에 집중하는 경향이 있습니다.

In [ ]:
@torch.no_grad()
def plot_attention(model, seq, layer=-1):
    model.eval()
    x = torch.tensor([seq], device=device)
    _, attns = model(x)
    A = attns[layer][0].cpu()                      # (heads, L, L)
    n_heads = A.shape[0]
    fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))
    for h in range(n_heads):
        ax = axes[h]
        ax.imshow(A[h], cmap="Blues", vmin=0)
        ax.set_xticks(range(len(seq)), seq)
        ax.set_yticks(range(len(seq)), sorted(seq))
        ax.set_xlabel("input token (key)")
        ax.set_title(f"layer {layer}, head {h}")
    axes[0].set_ylabel("output = sorted value (query)")
    plt.tight_layout()
    plt.show()

plot_attention(model_pe, test_seq)

## 8. PyTorch 내장 모듈과 비교

실제로는 `nn.TransformerEncoderLayer` / `nn.TransformerEncoder`를 사용하면 됩니다.
내부 구조는 위에서 만든 `EncoderBlock`과 같습니다 (Multi-Head Attention + Add & Norm + FFN).
단, PyTorch에는 Positional Encoding이 포함되어 있지 않으므로 직접 더해 주어야 합니다.

In [ ]:
class TorchTransformerSorter(nn.Module):
    def __init__(self, vocab_size=10, d_model=64, n_heads=4, d_ff=128, n_layers=2):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = SinusoidalPositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, tokens):
        x = self.pos(self.embed(tokens) * math.sqrt(self.d_model))
        return self.head(self.encoder(x)), None


torch.manual_seed(0)
torch_model = TorchTransformerSorter().to(device)
opt = torch.optim.Adam(torch_model.parameters(), lr=1e-3)
for step in range(1, 1501):
    torch_model.train()
    x, y = make_batch(256)
    loss = F.cross_entropy(torch_model(x)[0].reshape(-1, VOCAB), y.reshape(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        acc_tok, acc_seq = evaluate(torch_model)
        print(f"[nn.TransformerEncoder] step {step}  token acc {acc_tok:.3f}  sequence acc {acc_seq:.3f}")

## 9. 연습 문제 (Exercises)

1. **Causal mask**: `scaled_dot_product_attention`에 `mask = torch.triu(torch.ones(L, L, dtype=torch.bool), 1)`
   를 넣으면 각 위치가 미래(오른쪽)를 볼 수 없습니다 (decoder의 masked self-attention).
   정렬 문제에 causal mask를 쓰면 정확도가 어떻게 변할까요? 왜 그럴까요?
2. **Learned positional encoding**: `SinusoidalPositionalEncoding` 대신
   `nn.Embedding(max_len, d_model)`로 위치 벡터를 학습하도록 바꾸고 결과를 비교해 보세요.
3. **길이 일반화**: 길이 10으로 학습한 모델을 길이 15 수열에 적용하면 어떻게 될까요?
   (sinusoidal PE는 임의의 위치에 대해 정의되지만, 실제로 잘 동작하는지는 별개의 문제입니다.)
4. **Head / layer 수**: `n_heads=1`, `n_layers=1`로 줄이면 학습 속도와 최종 정확도가 어떻게 달라지나요?
5. **뒤집기 (reverse) 과제**: target을 `x.flip(1)`로 바꾸어 수열 뒤집기를 학습시켜 보세요.
   이 과제는 오직 위치 정보만으로 풀어야 하는 문제입니다.